In [ ]:
import torch
import numpy as np

In [ ]:
from os import W_OK
def softmax(x):
  e = np.exp(x - np.max(x))
  return e/e.sum(axis = 0)

def self_attention(Q,K,V, H = 4, mask = None):

  d_k = Q.size(-1) #Q is after embedding
  W_query = torch.zeros(d_k, d_k)
  W_key = torch.zeros(d_k, d_k)
  W_value = torch.zeros(d_k, d_k)
  Q = Q @ W_query
  K = K @ W_query
  V = V @ W_query
  Q_K_logits = Q @ K.transpose(-2,-1) / torch.sqrt(d_k)

  if mask ==


batch, seq_len, d_k = 2, 5, 8
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

3

In [ ]:
import torch

# Create a 2x3 tensor
x = torch.tensor([[[1, 2, 3,4, 5, 6], [5,5,5,5,5,5]]])
print(x.view(1,2,2,3).squeeze())

tensor([[[1, 2, 3],
         [4, 5, 6]],

        [[5, 5, 5],
         [5, 5, 5]]])


In [ ]:
torch.ones(4,4)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [ ]:
import torch.nn.functional as F
import torch.nn as nn
import torch

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, d_model, mask = None):
    super().__init__()

    self.head = num_heads
    self.d_k = d_model // num_heads
    self.mask = mask

    self.wQ = nn.Linear(d_model, d_model, bias = False)
    self.wK = nn.Linear(d_model, d_model, bias = False)
    self.wV = nn.Linear(d_model, d_model, bias = False)

    self.FFNN = nn.Sequential(
        nn.Linear(d_model, 2048),
        nn.ReLU(),
        nn.Linear(2048, d_model)
    )

  def head_splitter(self, x):
    batch, seq, _ = x.shape # batch, seq, 512
    x = x.view(batch, seq, self.head, self.d_k)
    x = x.transpose(1,2)
    return x

  def forward(self,x):
    batch, seq, _ = x.shape
    Q = self.head_splitter(self.wQ(x)) # batch, head, seq, d_k
    K = self.head_splitter(self.wK(x))
    V = self.head_splitter(self.wV(x))

    similarity_logit = (Q @ K.transpose(-2,-1)) /(self.d_k ** 0.5) # batch, head, seq, seq

    if self.mask == 'casual':
      mask = torch.tril(torch.ones(seq, seq)).bool().to(x.device)
      similarity_with_mask = similarity_logit.masked_fill(~mask, float('-inf'))
    else:
      similarity_with_mask = similarity_logit

    weights = F.softmax(similarity_with_mask, dim = -1)
    out = weights @ V
    out = out.transpose(1,2).contiguous().view(batch, seq, -1)
    out = self.FFNN(out)
    return out, weights


In [ ]:
x = torch.randn(2, 5, 512)
mha = MultiHeadAttention(num_heads = 4, d_model = 512, mask = 'casual')
out, weights = mha(x)
print(weights)
print(weights.shape)

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.5535, 0.4465, 0.0000, 0.0000, 0.0000],
          [0.4162, 0.2689, 0.3148, 0.0000, 0.0000],
          [0.1774, 0.2168, 0.2266, 0.3792, 0.0000],
          [0.1766, 0.2020, 0.2314, 0.1920, 0.1980]],

         [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.5813, 0.4187, 0.0000, 0.0000, 0.0000],
          [0.5329, 0.3040, 0.1631, 0.0000, 0.0000],
          [0.3207, 0.2439, 0.1151, 0.3202, 0.0000],
          [0.2132, 0.2365, 0.2140, 0.1452, 0.1912]],

         [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.5533, 0.4467, 0.0000, 0.0000, 0.0000],
          [0.3475, 0.4912, 0.1614, 0.0000, 0.0000],
          [0.2863, 0.2922, 0.2722, 0.1493, 0.0000],
          [0.1380, 0.1252, 0.1665, 0.3033, 0.2670]],

         [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.5032, 0.4968, 0.0000, 0.0000, 0.0000],
          [0.4971, 0.2124, 0.2905, 0.0000, 0.0000],
          [0.2270, 0.3982, 0.1459, 0.2289, 0.0000],
      